In [1]:
pip install ir-datasets

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install pyautocorpus

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 379.9/379.9 kB 8.2 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [1]:
import ir_datasets

dataset = ir_datasets.load("trec-fair-2021")

[INFO] acessing TREC Fair Ranking 2021 through trec-fair-2021 is deprecated; use trec-fair/2021 instead.


In [8]:
import pandas as pd
from tqdm import tqdm
corpus_dict = {"docno": [], "title": [], "text": [], "geographic_locations": []}
for doc in tqdm(dataset.docs_iter()):
    doc_id = doc[0]
    title = doc[1]
    article = doc[2]
    geo = doc[6]

    corpus_dict["docno"].append(int(doc_id))
    corpus_dict["text"].append(str(article))
    corpus_dict["title"].append(str(title))
    corpus_dict["geographic_locations"].append(geo)
    
corpus_df = pd.DataFrame(corpus_dict)
corpus_df.to_csv("./trec-2021/trec_corpus.csv", index=False)

6280328it [04:13, 24749.63it/s]


In [9]:
# this cell is to constuct a dataframe which contains two columns query_id and rel_doc_id_list.

import ir_datasets
import pandas as pd

query_rel_dict = dict()

dataset = ir_datasets.load("trec-fair-2021/train")
for i, qrel in enumerate(dataset.qrels_iter()):
    # qrel # namedtuple<query_id, doc_id, relevance, iteration>
    if qrel.relevance==1:
        if qrel.query_id not in query_rel_dict:
            query_rel_dict[qrel.query_id] = []
        query_rel_dict[qrel.query_id].append(qrel.doc_id)

# Convert the dictionary to a list of rows
rows = [{"query_id": k, "rel_doc_id_list": v} for k, v in query_rel_dict.items()]

# Create a DataFrame
qrel_df = pd.DataFrame(rows, columns=["query_id", "rel_doc_id_list"])

# View the DataFrame
print(qrel_df.head())

[INFO] acessing TREC Fair Ranking 2021 through trec-fair-2021/train is deprecated; use trec-fair/2021/train instead.


  query_id                                    rel_doc_id_list
0        1  [572, 627, 903, 1193, 1542, 1634, 3751, 3866, ...
1        2  [682, 954, 1170, 1315, 1322, 1324, 1325, 1435,...
2        3  [5729, 8490, 9623, 10391, 12231, 13791, 16078,...
3        4  [849, 852, 1293, 1902, 1942, 2039, 2075, 2082,...
4        5  [1135, 1136, 1293, 1893, 2129, 2140, 3797, 380...


In [10]:
len(qrel_df)

57

In [15]:
# # this cell is to get id of documents containing less than 250 words.
# import gzip
# import json
# from tqdm import tqdm

# desired_doc_ids = []
# sum_length = 0
# max_body_length_treshold = 512
# min_body_length_treshold = 0
# min_title_length_treshold = 0
# with gzip.open('./trec-2021/trec_corpus.json.gz', 'rt', encoding='utf-8') as f:
#     for line in tqdm(f):
#         doc = json.loads(line)
#         print(doc.keys())
#         break
#         if (len(doc["text"].split()) <= max_body_length_treshold): #(len(doc["plain"].split()) >= min_body_length_treshold) and (len(doc["title"].split()) >= min_title_length_treshold)
#             sum_length += len(doc["text"].split())
#             desired_doc_ids.append(int(doc["id"]))

#         # desired_doc_ids.append(doc["id"])
# this cell is to get id of documents containing less than 250 words.
from tqdm import tqdm
import pandas as pd

desired_doc_ids = []
sum_length = 0
max_body_length_treshold = 512
min_body_length_treshold = 0
min_title_length_treshold = 0
# corpus_df = pd.read_csv("./trec-2021/trec_corpus.csv")
for eachrow in tqdm(corpus_df.itertuples(index=False)):
    article = eachrow.text
    doc_id = eachrow.docno
    article_len = len(article.split())
    if article_len <= max_body_length_treshold: #(len(doc["plain"].split()) >= min_body_length_treshold) and (len(doc["title"].split()) >= min_title_length_treshold)
        sum_length += article_len
        desired_doc_ids.append(int(doc_id))

6280328it [01:23, 75015.81it/s] 


In [16]:
len(desired_doc_ids), sum_length/len(desired_doc_ids)

(4905140, 156.4988387691279)

In [17]:
for i in desired_doc_ids:
    print(type(i))
    break

<class 'int'>


In [18]:
# This cell is to filter out documents not being the list of desired doc ids.
import gzip
import json
from tqdm import tqdm

desired_doc_ids_set = set(desired_doc_ids)  # ensures fast lookup
filtered_desired_doc_ids = []

with gzip.open('./trec-2021/trec_metadata.json.gz', 'rt', encoding='utf-8') as f:
    for line in tqdm(f, desc="Filtering docs"):
        data = json.loads(line)
        # print(data.keys())
        # print(data["geographic_locations"])
        if (int(data["page_id"]) in desired_doc_ids_set) and (data["geographic_locations"] != ""):
            filtered_desired_doc_ids.append(data["page_id"])

Filtering docs: 6023428it [00:13, 448132.91it/s]


In [19]:
len(filtered_desired_doc_ids)

4692610

In [20]:
with open("./trec-2021/filtered_desired_doc_ids.txt", "w") as f:
    for doc_id in filtered_desired_doc_ids:
        f.write(str(doc_id) + "\n")

In [21]:
filtered_desired_doc_ids = []
with open("./trec-2021/filtered_desired_doc_ids.txt", "r") as f:
    for line in f:
        filtered_desired_doc_ids.append(int(line.strip()))

In [22]:
# Ensure your list of desired IDs is a set for faster lookup
filtered_desired_doc_ids_set = set(filtered_desired_doc_ids)
def selected_docs(row):
    selected_docs_list = []
    for doc_id in row["rel_doc_id_list"]:
        if int(doc_id) in filtered_desired_doc_ids_set:
            selected_docs_list.append(int(doc_id))
    return selected_docs_list

# Create a new column to count how many relevant documents per query are in desired_id_set
qrel_df["selected_docs_list"] = qrel_df.apply(selected_docs, axis=1)

# Optionally: preview the result
print(qrel_df[["query_id", "selected_docs_list"]].head())

  query_id                                 selected_docs_list
0        1  [36834, 42646, 45847, 49396, 55751, 56122, 623...
1        2  [1873, 5970, 6298, 7118, 12483, 12821, 13773, ...
2        3  [16546, 22104, 22355, 22356, 31477, 49132, 598...
3        4  [13463, 14315, 15365, 15536, 16941, 17869, 200...
4        5  [1136, 3800, 3805, 3814, 3851, 28416, 37221, 4...


In [23]:
# Ensure your list of desired IDs is a set for faster lookup
def count_docs(row):
    return len(row["selected_docs_list"])

# Create a new column to count how many relevant documents per query are in desired_id_set
qrel_df["desired_doc_count"] = qrel_df.apply(count_docs, axis=1)

In [24]:
qrel_df

,query_id,rel_doc_id_list,selected_docs_list,desired_doc_count
0,1,"[572, 627, 903, 1193, 1542, 1634, 3751, 3866, ...","[36834, 42646, 45847, 49396, 55751, 56122, 623...",4217
1,2,"[682, 954, 1170, 1315, 1322, 1324, 1325, 1435,...","[1873, 5970, 6298, 7118, 12483, 12821, 13773, ...",42915
2,3,"[5729, 8490, 9623, 10391, 12231, 13791, 16078,...","[16546, 22104, 22355, 22356, 31477, 49132, 598...",50928
3,4,"[849, 852, 1293, 1902, 1942, 2039, 2075, 2082,...","[13463, 14315, 15365, 15536, 16941, 17869, 200...",36472
4,5,"[1135, 1136, 1293, 1893, 2129, 2140, 3797, 380...","[1136, 3800, 3805, 3814, 3851, 28416, 37221, 4...",34002
5,6,"[308, 340, 700, 728, 736, 784, 851, 852, 892, ...","[340, 892, 910, 1563, 1654, 1751, 1763, 1873, ...",65343
6,7,"[595, 890, 1020, 1135, 1293, 1806, 1893, 2575,...","[10620, 12556, 16546, 22104, 30097, 31477, 367...",495556
7,8,"[344, 676, 808, 872, 1247, 1806, 1828, 2083, 2...","[2723, 3785, 6337, 7033, 8079, 8436, 10275, 13...",52379
8,9,"[4243, 17163, 25506, 39027, 45360, 45772, 4811...","[69190, 80142, 84775, 87963, 90653, 96901, 106...",10202
9,10,"[1017, 1178, 1239, 1669, 1735, 2642, 2784, 284...","[1669, 8676, 60347, 94782, 99570, 100099, 1002...",2979


In [25]:
top_3 = qrel_df.nlargest(3, 'desired_doc_count')
top_val = qrel_df['desired_doc_count'].nlargest(3).min()
qrel_df = qrel_df[qrel_df['desired_doc_count'] >= top_val]

In [26]:
qrel_df

,query_id,rel_doc_id_list,selected_docs_list,desired_doc_count
6,7,"[595, 890, 1020, 1135, 1293, 1806, 1893, 2575,...","[10620, 12556, 16546, 22104, 30097, 31477, 367...",495556
22,23,"[925, 2174, 2273, 2289, 2358, 3165, 4224, 4392...","[8649, 10830, 11052, 30097, 44237, 45316, 5226...",248721
40,41,"[595, 844, 890, 1216, 3138, 3354, 4405, 4443, ...","[5038, 15495, 16546, 22104, 22355, 22356, 2236...",149455


In [27]:
import ir_datasets
dataset = ir_datasets.load("trec-fair-2021/train")
for query in dataset.queries_iter():
    if query[0] in qrel_df["query_id"].tolist(): # namedtuple<query_id, text, url>
      print(query)

FairTrecQuery(query_id='7', text='Biography/sports and games work group', keywords=['sports biography', 'athlets', 'player', 'memoir', 'refree', 'coach'], scope='The Sports and Games Work Group is a working group of members of the Biography WikiProject dedicated to ensuring quality and coverage of biography articles.', homepage='https://en.wikipedia.org/wiki/Wikipedia:WikiProject_Biography/Sports_and_games')
FairTrecQuery(query_id='23', text='Football', keywords=['football', 'soccer', 'team', 'club', 'league', 'season', 'championship', 'match', 'player', 'stadium', 'pitch'], scope='Any article relating to association football (soccer) for example including football clubs, national teams, national associations, competitions, leagues, seasons, matches, players, stadiums and governing bodies such as FIFA and UEFA.', homepage='https://en.wikipedia.org/wiki/Wikipedia:WikiProject_Football')
FairTrecQuery(query_id='41', text='Olympics', keywords=['olympic games', 'paralympics', 'olympic medal

[INFO] acessing TREC Fair Ranking 2021 through trec-fair-2021/train is deprecated; use trec-fair/2021/train instead.


In [32]:
final_chosen_documents = set()
for l in qrel_df["selected_docs_list"].tolist():
    final_chosen_documents.update(set(l))

In [33]:
len(final_chosen_documents)

604013

In [34]:
import gzip
import json
import pandas as pd
from tqdm import tqdm

rows = []
investigated_documents = set()
with gzip.open('./trec-2021/trec_metadata.json.gz', 'rt', encoding='utf-8') as f:
    for line in tqdm(f, desc="Filtering docs"):
        data = json.loads(line)
        if int(data["page_id"]) in final_chosen_documents:
            if int(data["page_id"]) in investigated_documents:
                continue
            investigated_documents.add(int(data["page_id"]))
            if data["geographic_locations"] == []:
                geographic_locations = ["None"]
            else:
                geographic_locations = data["geographic_locations"]
            rows.append({
                "docno": int(data["page_id"]),
                "geographic_locations": geographic_locations,
            })
# Create DataFrame once
document_features = pd.DataFrame(rows)

Filtering docs: 6023428it [00:13, 434051.04it/s]


In [35]:
len(document_features)

604013

In [36]:
all_geographic_locations = set()
unique_geographic_location_lists = set()
for geographic_locations in document_features['geographic_locations'].tolist():
    all_geographic_locations.update(set(geographic_locations))
    unique_geographic_location_lists.add(str(geographic_locations))

In [37]:
len(all_geographic_locations), all_geographic_locations

(7,
 {'Africa',
  'Asia',
  'Europe',
  'Latin America and the Caribbean',
  'None',
  'Northern America',
  'Oceania'})

In [30]:
len(unique_geographic_location_lists), unique_geographic_location_lists

(31,
 {"['Africa']",
  "['Asia', 'Africa']",
  "['Asia', 'Europe', 'Africa']",
  "['Asia', 'Europe', 'Northern America']",
  "['Asia', 'Europe']",
  "['Asia', 'Northern America']",
  "['Asia']",
  "['Europe', 'Africa']",
  "['Europe']",
  "['Latin America and the Caribbean', 'Africa']",
  "['Latin America and the Caribbean', 'Asia', 'Europe']",
  "['Latin America and the Caribbean', 'Asia', 'Northern America']",
  "['Latin America and the Caribbean', 'Asia']",
  "['Latin America and the Caribbean', 'Europe']",
  "['Latin America and the Caribbean', 'Northern America', 'Europe']",
  "['Latin America and the Caribbean', 'Northern America']",
  "['Latin America and the Caribbean', 'Oceania']",
  "['Latin America and the Caribbean']",
  "['None']",
  "['Northern America', 'Africa']",
  "['Northern America', 'Asia']",
  "['Northern America', 'Europe', 'Asia']",
  "['Northern America', 'Europe']",
  "['Northern America']",
  "['Oceania', 'Africa']",
  "['Oceania', 'Asia', 'Europe']",
  "['Oc

In [38]:
import ast
geo_counter = dict()
for str_list in unique_geographic_location_lists:
    geo_list = ast.literal_eval(str_list)
    for geo in geo_list:
        if geo in geo_counter:
            geo_counter[geo] += 1
            continue
        geo_counter[geo] = 1
geo_counter

{'Oceania': 11,
 'Asia': 17,
 'Europe': 17,
 'Africa': 8,
 'Northern America': 18,
 'Latin America and the Caribbean': 12,
 'None': 1}

In [39]:
import ast
def calucluate_represnetative_per_combination(unique_geographic_location_lists): 
    geo_counter = dict()
    for str_list in unique_geographic_location_lists:
        geo_list = ast.literal_eval(str_list)
        for geo in geo_list:
            if geo in geo_counter:
                geo_counter[geo] += 1
                continue
            geo_counter[geo] = 1

    max_value = max(geo_counter.values())
    geo_counter = {k: max_value-(v-1) for k, v in geo_counter.items()}
    return geo_counter

In [40]:
import ast

def select_one_row_per_combination_fast(df, final_df):
    selected_rows = []

    # Create a set of hashes for rows in final_df to check for duplicates efficiently
    existing_hashes = set(final_df.astype(str).apply(lambda row: hash(tuple(row)), axis=1))

    df['geographic_locations_str'] = df['geographic_locations'].apply(lambda x: str(x))
    category_columns = ['geographic_locations_str']
    grouped = df.groupby(category_columns, sort=False)
    unique_geographic_location_lists = []
    for geographic_locations_str, group in grouped:
        # print(type(geographic_locations_str[0]))
        unique_geographic_location_lists.append(geographic_locations_str[0])
    # print(unique_geographic_location_lists)
    geo_dict = calucluate_represnetative_per_combination(unique_geographic_location_lists)
    
    for geographic_locations_str, group in grouped:
        # Shuffle rows in group (you can set random_state if you want reproducibility)
        group_shuffled = group.sample(frac=1, random_state=42)

        # Try each row until we find one not in final_df
        if len(ast.literal_eval(geographic_locations_str[0])) == 1:
            # print(ast.literal_eval(geographic_locations_str[0]))
            number_of_representatives = geo_dict[ast.literal_eval(geographic_locations_str[0])[0]]
            # print(number_of_representatives)
            # print(len(group_shuffled))
        else:
            number_of_representatives = 1

        selected_counter = 0 
        for _, row in group_shuffled.iterrows():
            row_hash = hash(tuple(row.astype(str)))
            # if row_hash not in existing_hashes:
            selected_rows.append(row)
            # existing_hashes.add(row_hash)
            selected_counter += 1
            if selected_counter == number_of_representatives:
                break
    # Return the selected rows as a DataFrame
    return pd.DataFrame(selected_rows)

In [41]:
final_df = pd.DataFrame()  # Initialize an empty DataFrame to collect results
count = 0
for eachrow in qrel_df.itertuples(index=False):
    selected_docs_list = eachrow.selected_docs_list
    print(count)
    filtered_df = document_features[document_features['docno'].isin(selected_docs_list)]
    selected_df = select_one_row_per_combination_fast(filtered_df, final_df)
    print(f"Selected {len(selected_df)} rows.")
    
    all_geographic_locations = set()
    geo_counter = dict()
    for geographic_locations in selected_df['geographic_locations'].tolist():
        for geo_location in geographic_locations:
            if geo_location in geo_counter:
                geo_counter[geo_location] += 1
                continue
            geo_counter[geo_location] = 1
    print(geo_counter)
    print("***********************")
    selected_df.to_csv(f"./trec-2021/query_ids_{eachrow.query_id}.csv", index=False)
    # print(selected_df.head())
    final_df = pd.concat([final_df, selected_df], ignore_index=True)
    count += 1

0


/tmp/ipykernel_366/864522898.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['geographic_locations_str'] = df['geographic_locations'].apply(lambda x: str(x))


Selected 80 rows.
{'None': 18, 'Northern America': 18, 'Africa': 18, 'Europe': 18, 'Asia': 18, 'Oceania': 18, 'Latin America and the Caribbean': 18}
***********************
1


/tmp/ipykernel_366/864522898.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['geographic_locations_str'] = df['geographic_locations'].apply(lambda x: str(x))
/tmp/ipykernel_366/864522898.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['geographic_locations_str'] = df['geographic_locations'].apply(lambda x: str(x))


Selected 52 rows.
{'Europe': 12, 'None': 12, 'Latin America and the Caribbean': 12, 'Africa': 12, 'Northern America': 12, 'Asia': 12, 'Oceania': 12}
***********************
2
Selected 66 rows.
{'None': 14, 'Africa': 14, 'Europe': 14, 'Northern America': 14, 'Asia': 14, 'Oceania': 14, 'Latin America and the Caribbean': 14}
***********************


In [42]:
import gzip
import json
from tqdm import tqdm
import pandas as pd

corpus_dict = {"docno": [], "title": [], "text": []}
with gzip.open('./trec-2021/trec_corpus.json.gz', 'rt', encoding='utf-8') as f:
    for line in tqdm(f):
        doc = json.loads(line)
        if int(doc["id"]) in final_chosen_documents:
              corpus_dict["docno"].append(int(doc["id"]))
              corpus_dict["text"].append(doc["text"])
              corpus_dict["title"].append(doc["title"])

corpus_df = pd.DataFrame(corpus_dict)
corpus_df.to_csv("./trec-2021/corpus.csv", index=False)

6280328it [06:28, 16164.75it/s]


In [43]:
document_features.to_csv("./trec-2021/document_features.csv", index=False)